# 11.04 -- Production Observability: Metrics, Tracing, and Alerting

**Module:** 11 -- LLM Systems Engineering  
**Notebook:** 4 of 5  
**Prerequisites:** 09.03 (Token Streaming), 09.05 (LLM Serving), 10.05 (Inference Benchmarking)

---

## Learning Objectives

1. Design a complete observability stack for an LLM serving system
2. Implement structured logging with request tracing and correlation IDs
3. Build a metrics collector tracking TTFT, TPOT, throughput, and error rates
4. Implement SLO tracking with budget burn rates and alerting thresholds
5. Build a health-check endpoint and understand its role in load balancers

---

## 1. Why Observability Is Different for LLMs

LLM serving has observability challenges that do not arise in typical APIs:

**Variable latency**: a 10-token response completes in ~0.5 s; a 2000-token response
takes 40-100 s. Standard latency percentiles conflate these into a misleading average.
You need TTFT (first token) and TPOT (per-token rate) as separate metrics.

**Long-tail requests**: a single request generating 10,000 tokens can saturate a
GPU slot for minutes. Tracking the request queue depth and per-slot occupancy is
essential to detect and shed load before SLOs are breached.

**Cost-per-token visibility**: unlike fixed-cost APIs, LLM serving cost scales
with output tokens generated. Tracking tokens-in, tokens-out per request is
required for both cost allocation and detecting runaway generation loops.

**Quality degradation**: a server can be 'up' (returning 200 OK) while producing
garbage outputs (repeated tokens, truncated responses). Output quality metrics
like response length distribution and repetition rate should be tracked alongside
latency and error rate.

---

## 2. The Observability Stack

```
  Application layer:
    RequestLogger         -- structured JSON log per request (id, latency, tokens)
    MetricsCollector      -- aggregated counters and histograms
    TracingMiddleware     -- distributed trace context propagation

  Infrastructure layer:
    Prometheus            -- time-series metrics scraping
    Grafana               -- dashboarding and alerting
    Jaeger / Tempo        -- distributed tracing storage
    ELK / Loki            -- log aggregation and search

  SLO layer:
    SLOTracker            -- burn rate calculation and alerting
    HealthCheck           -- load balancer integration
```

This notebook implements the application layer from scratch and simulates
the infrastructure layer with in-memory stores.


In [ ]:
# Environment setup.
#
# We implement the full observability stack in pure Python.
# No external monitoring infrastructure is required -- all stores are in-memory.
# The interfaces are designed to be drop-in replaceable with Prometheus,
# Jaeger, or any other production monitoring system.

import time
import uuid
import json
import math
import random
import threading
import statistics
from collections import defaultdict, deque
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Callable, Any
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
random.seed(42)

print('Observability stack initialising...')


## 3. Structured Request Logging


In [ ]:
# Structured request logger.
#
# Every LLM request should produce a single structured JSON log entry.
# Structured logs are machine-readable (unlike free-form text logs) and
# can be queried, aggregated, and alerted on by log management systems.
#
# Each log entry contains:
#   request_id:     UUID for end-to-end tracing across services
#   trace_id:       distributed trace ID (propagated from the HTTP request)
#   model:          model name and version (for A/B testing visibility)
#   prompt_tokens:  input token count (cost and latency predictor)
#   output_tokens:  output token count (actual cost incurred)
#   ttft_ms:        time-to-first-token (user-perceived latency)
#   tpot_ms:        time-per-output-token (streaming smoothness)
#   e2e_ms:         total request duration
#   status:         success / error / timeout
#   error_type:     if status != success, the error category
#   user_id:        for per-user analytics and abuse detection
#   finish_reason:  'stop', 'length', 'content_filter', 'error'

@dataclass
class RequestLog:
    request_id:    str
    trace_id:      str
    model:         str
    prompt_tokens: int
    output_tokens: int
    ttft_ms:       float
    tpot_ms:       float
    e2e_ms:        float
    status:        str     # 'success', 'error', 'timeout'
    finish_reason: str     # 'stop', 'length', 'error'
    user_id:       Optional[str] = None
    error_type:    Optional[str] = None
    metadata:      Dict   = field(default_factory=dict)
    timestamp:     float  = field(default_factory=time.time)

    def to_json(self) -> str:
        return json.dumps({
            'ts':           self.timestamp,
            'request_id':   self.request_id,
            'trace_id':     self.trace_id,
            'model':        self.model,
            'prompt_tokens':self.prompt_tokens,
            'output_tokens':self.output_tokens,
            'ttft_ms':      round(self.ttft_ms, 2),
            'tpot_ms':      round(self.tpot_ms, 2),
            'e2e_ms':       round(self.e2e_ms, 2),
            'status':       self.status,
            'finish_reason':self.finish_reason,
            'user_id':      self.user_id,
            'error_type':   self.error_type,
        })


class RequestLogger:
    """
    Collects request logs; in production these would be emitted to stdout
    (for collection by a log aggregator) or written to a file sink.
    """

    def __init__(self, max_buffer: int = 10_000):
        self._logs: deque = deque(maxlen=max_buffer)
        self._lock = threading.Lock()

    def log(self, entry: RequestLog):
        with self._lock:
            self._logs.append(entry)
        # In production: print(entry.to_json(), flush=True)

    @property
    def logs(self) -> List[RequestLog]:
        with self._lock:
            return list(self._logs)


# Simulate a realistic request workload
def simulate_request(
    model:        str = 'llama-3-8b',
    error_rate:   float = 0.02,
    timeout_rate: float = 0.01,
) -> RequestLog:
    """
    Generate a realistic synthetic request log entry.

    Uses empirical distributions:
      - Prompt tokens: log-normal (median ~120 tokens, occasional long prompts)
      - Output tokens: log-normal (median ~80 tokens, heavy tail for long responses)
      - TTFT: 50-300 ms (scales with prompt length)
      - TPOT: 15-30 ms/token (varies with load)
    """
    prompt_tokens = max(10, int(np.random.lognormal(4.8, 0.8)))
    output_tokens = max(1,  int(np.random.lognormal(4.2, 1.0)))

    ttft_ms = 40 + prompt_tokens * 0.15 + np.random.exponential(20)
    tpot_ms = 18 + np.random.exponential(5)
    e2e_ms  = ttft_ms + tpot_ms * output_tokens

    r = random.random()
    if r < timeout_rate:
        status       = 'timeout'
        finish_reason = 'error'
        error_type   = 'timeout'
        output_tokens = 0
        e2e_ms       = 30_000
    elif r < timeout_rate + error_rate:
        status        = 'error'
        finish_reason = 'error'
        error_type    = random.choice(['oom', 'context_length', 'invalid_request'])
        output_tokens = 0
    else:
        status        = 'success'
        finish_reason = 'length' if output_tokens > 1000 else 'stop'
        error_type    = None

    return RequestLog(
        request_id    = str(uuid.uuid4())[:8],
        trace_id      = str(uuid.uuid4())[:16],
        model         = model,
        prompt_tokens = prompt_tokens,
        output_tokens = output_tokens,
        ttft_ms       = ttft_ms,
        tpot_ms       = tpot_ms,
        e2e_ms        = e2e_ms,
        status        = status,
        finish_reason = finish_reason,
        user_id       = f'user_{random.randint(1, 200)}',
        error_type    = error_type,
    )


# Generate 500 requests
logger = RequestLogger()
for _ in range(500):
    logger.log(simulate_request())

logs = logger.logs
successes = [l for l in logs if l.status == 'success']
print(f'Simulated {len(logs)} requests:')
print(f'  Success:  {sum(1 for l in logs if l.status=="success"):>5}')
print(f'  Error:    {sum(1 for l in logs if l.status=="error"):>5}')
print(f'  Timeout:  {sum(1 for l in logs if l.status=="timeout"):>5}')
print()
print('Sample log entry (JSON):')
print(successes[0].to_json())


In [ ]:
# Metrics collector with histogram support.
#
# A production metrics system needs two types of measurements:
#
#   Counters: monotonically increasing values (total requests, total tokens,
#             total errors). Used to compute rates: req/s, tokens/s, error rate.
#
#   Histograms: distribution of values (latency, token counts).
#     Store in configurable buckets (e.g. [10, 50, 100, 250, 500, 1000, 5000] ms).
#     Prometheus histograms support efficient P50/P95/P99 queries.
#
# We also implement a sliding-window rate calculator for real-time metrics:
# request rate and error rate over the last 60 seconds, updated each second.
#
# The MetricsCollector is the central object that a FastAPI middleware would
# update on every request completion. A /metrics endpoint exposes the current
# state in Prometheus text format for scraping.

class Histogram:
    """
    Stores a distribution as bucket counts.
    Buckets are upper bounds; the last bucket is +inf.
    """

    def __init__(self, buckets: List[float]):
        self.buckets   = sorted(buckets) + [float('inf')]
        self.counts    = [0] * len(self.buckets)
        self.total     = 0
        self._samples: List[float] = []   # kept for percentile computation

    def observe(self, value: float):
        self._samples.append(value)
        self.total += 1
        for i, upper in enumerate(self.buckets):
            if value <= upper:
                self.counts[i] += 1
                break

    def percentile(self, p: float) -> float:
        if not self._samples:
            return 0.0
        return float(np.percentile(self._samples, p))


class MetricsCollector:
    """
    Central metrics store for an LLM serving system.

    Tracks:
      - Request counts by status
      - Token counts (input and output)
      - Latency histograms: TTFT, TPOT, E2E
      - Sliding window rates: requests/s, errors/s, tokens/s
    """

    TTFT_BUCKETS = [20, 50, 100, 200, 500, 1000, 2000, 5000]
    TPOT_BUCKETS = [5, 10, 20, 30, 50, 100, 200]
    E2E_BUCKETS  = [100, 500, 1000, 2000, 5000, 10000, 30000, 60000]

    def __init__(self):
        self.total_requests    = defaultdict(int)   # keyed by status
        self.total_prompt_toks = 0
        self.total_output_toks = 0
        self.total_errors      = defaultdict(int)   # keyed by error_type

        self.ttft_hist  = Histogram(self.TTFT_BUCKETS)
        self.tpot_hist  = Histogram(self.TPOT_BUCKETS)
        self.e2e_hist   = Histogram(self.E2E_BUCKETS)

        # Sliding window: (timestamp, n_requests, n_errors, n_tokens)
        self._window:  deque = deque()
        self._lock = threading.Lock()

    def record(self, log: RequestLog):
        with self._lock:
            self.total_requests[log.status] += 1
            self.total_prompt_toks += log.prompt_tokens
            self.total_output_toks += log.output_tokens

            if log.error_type:
                self.total_errors[log.error_type] += 1

            if log.status == 'success':
                self.ttft_hist.observe(log.ttft_ms)
                self.tpot_hist.observe(log.tpot_ms)
                self.e2e_hist.observe(log.e2e_ms)

            self._window.append((log.timestamp, 1,
                                  1 if log.status != 'success' else 0,
                                  log.output_tokens))

    def windowed_rate(self, window_s: float = 60.0) -> Dict:
        """Compute request rate, error rate, and token rate over the last window_s seconds."""
        now = time.time()
        cutoff = now - window_s
        with self._lock:
            # Prune old entries
            while self._window and self._window[0][0] < cutoff:
                self._window.popleft()
            reqs   = sum(e[1] for e in self._window)
            errors = sum(e[2] for e in self._window)
            tokens = sum(e[3] for e in self._window)
        return {
            'rps':         reqs / window_s,
            'error_rate':  errors / max(reqs, 1),
            'tokens_per_s': tokens / window_s,
        }

    def summary(self) -> Dict:
        total = sum(self.total_requests.values())
        return {
            'total_requests':   total,
            'error_rate':       self.total_requests.get('error', 0) / max(total, 1),
            'timeout_rate':     self.total_requests.get('timeout', 0) / max(total, 1),
            'ttft_p50_ms':      self.ttft_hist.percentile(50),
            'ttft_p99_ms':      self.ttft_hist.percentile(99),
            'tpot_p50_ms':      self.tpot_hist.percentile(50),
            'tpot_p99_ms':      self.tpot_hist.percentile(99),
            'e2e_p50_ms':       self.e2e_hist.percentile(50),
            'e2e_p99_ms':       self.e2e_hist.percentile(99),
            'total_output_toks': self.total_output_toks,
            'error_breakdown':  dict(self.total_errors),
        }


# Ingest all 500 simulated logs
collector = MetricsCollector()
for log in logs:
    log.timestamp = time.time() - random.uniform(0, 120)  # spread over last 2 min
    collector.record(log)

summary = collector.summary()
print('Metrics summary:')
for k, v in summary.items():
    if isinstance(v, float):
        print(f'  {k:<24}: {v:.2f}')
    else:
        print(f'  {k:<24}: {v}')


In [ ]:
# SLO tracking with error budget and burn rate alerting.
#
# An SLO (Service Level Objective) defines a reliability target:
#   'TTFT P99 < 500 ms for 99.9% of requests over any rolling 30-day window'
#
# Error budget: the fraction of requests that may violate the SLO.
#   99.9% availability -> 0.1% error budget = 43 minutes of downtime per month.
#
# Burn rate: how fast the error budget is being consumed relative to its
# allowable rate. A burn rate of 1.0 exactly exhausts the budget by end of window.
# A burn rate > 1.0 means the budget will be exhausted before the window ends.
#
# Alerting tiers:
#   Burn rate > 14: critical (budget exhausted in ~2 hours)  -> page on-call
#   Burn rate > 6:  warning  (budget exhausted in ~5 hours)  -> Slack alert
#   Burn rate > 1:  info     (budget tracking to exceed)     -> dashboard annotation
#
# This multi-window approach (Google SRE Workbook, Chapter 5) avoids both
# false positives (single slow minute triggering a page) and slow detection
# (gradual degradation not caught until the budget is nearly exhausted).

@dataclass
class SLODefinition:
    name:           str
    target_pct:     float   # e.g. 99.9 means 99.9% of requests must satisfy
    metric_fn:      Callable[[RequestLog], float]  # extract the metric value
    threshold:      float   # the value that must not be exceeded
    window_days:    int = 30

    @property
    def error_budget(self) -> float:
        return (100 - self.target_pct) / 100


class SLOTracker:
    """
    Tracks SLO compliance and computes burn rates for alerting.

    Maintains a rolling window of metric observations, computes the
    current SLO violation rate, and derives the burn rate.
    """

    def __init__(self, slo: SLODefinition, window_s: float = 3600):
        self.slo      = slo
        self.window_s = window_s   # monitoring window (1 hour for short-window alerts)
        self._events: deque = deque()   # (timestamp, violated: bool)

    def record(self, log: RequestLog):
        if log.status != 'success':
            return
        value    = self.slo.metric_fn(log)
        violated = value > self.slo.threshold
        self._events.append((log.timestamp, violated))

    def burn_rate(self) -> Dict:
        """
        Compute current burn rate and remaining error budget.

        Burn rate = (observed violation rate) / (allowed violation rate)
        where allowed violation rate = 1 - (target / 100).

        Returns a dict with burn_rate, violation_rate, remaining_budget,
        and an alert level.
        """
        now    = time.time()
        cutoff = now - self.window_s
        recent = [(ts, v) for ts, v in self._events if ts >= cutoff]

        if not recent:
            return {'burn_rate': 0, 'violation_rate': 0, 'alert': 'insufficient_data'}

        n_total    = len(recent)
        n_violated = sum(v for _, v in recent)
        viol_rate  = n_violated / n_total
        burn       = viol_rate / max(self.slo.error_budget, 1e-10)

        alert = 'ok'
        if   burn > 14: alert = 'critical'
        elif burn > 6:  alert = 'warning'
        elif burn > 1:  alert = 'info'

        return {
            'slo_name':        self.slo.name,
            'burn_rate':       round(burn, 2),
            'violation_rate':  round(viol_rate, 4),
            'error_budget':    round(self.slo.error_budget, 4),
            'n_evaluated':     n_total,
            'n_violated':      n_violated,
            'alert':           alert,
        }


# Define three SLOs
slos = [
    SLODefinition('ttft_p99',  99.0, lambda l: l.ttft_ms,  500),
    SLODefinition('tpot_p99',  99.0, lambda l: l.tpot_ms,   50),
    SLODefinition('e2e_p95',   95.0, lambda l: l.e2e_ms,  30_000),
]

trackers = [SLOTracker(slo, window_s=3600) for slo in slos]
for log in logs:
    for tracker in trackers:
        tracker.record(log)

print('SLO burn rates:')
for tracker in trackers:
    r = tracker.burn_rate()
    alert_emoji = {'ok': 'OK', 'info': 'INFO', 'warning': 'WARN', 'critical': 'CRIT'}
    level = alert_emoji.get(r['alert'], r['alert'])
    print(f'  [{level}]  {r["slo_name"]:<15}  '
          f'burn={r["burn_rate"]:>6.2f}  '
          f'violations={r["n_violated"]}/{r["n_evaluated"]}  '
          f'(budget={r["error_budget"]:.1%})')


In [ ]:
# Production observability dashboard.
#
# A production LLM observability dashboard shows six key views:
#   1. Request rate and error rate over time (health pulse)
#   2. TTFT P50/P99 over time (latency trend)
#   3. Output token distribution (cost and response quality signal)
#   4. SLO burn rate gauges (actionable alert state)
#   5. Error type breakdown (debugging aid)
#   6. Throughput in tokens/s over time (capacity planning)

# Simulate 4 hours of traffic for time-series plots
# Inject a degradation event at t=2h (doubled TTFT for 30 minutes)

hourly_logs = []
N_HOURS  = 4
INTERVAL = 60    # one log entry per second

t_start = time.time() - N_HOURS * 3600
for i in range(N_HOURS * 3600 // INTERVAL):
    t = t_start + i * INTERVAL
    # Inject degradation: 2h to 2.5h -> TTFT doubled
    degraded = (2 * 3600 <= i * INTERVAL < 2.5 * 3600)

    for _ in range(random.randint(3, 8)):  # variable request rate
        req = simulate_request(
            error_rate=0.05 if degraded else 0.02,
            timeout_rate=0.02 if degraded else 0.005,
        )
        req.timestamp = t + random.random() * INTERVAL
        if degraded and req.status == 'success':
            req.ttft_ms  *= 2.5   # simulate TTFT regression
            req.e2e_ms   *= 1.8
        hourly_logs.append(req)

# Build time-series by binning into 5-minute windows
BIN_S = 300
n_bins = N_HOURS * 3600 // BIN_S

bins_rps     = []
bins_err     = []
bins_ttft50  = []
bins_ttft99  = []
bins_tokens  = []

for b in range(n_bins):
    t_lo = t_start + b * BIN_S
    t_hi = t_lo + BIN_S
    bucket = [l for l in hourly_logs if t_lo <= l.timestamp < t_hi]
    succ   = [l for l in bucket if l.status == 'success']
    bins_rps.append(len(bucket) / BIN_S)
    bins_err.append(sum(1 for l in bucket if l.status != 'success') / max(len(bucket), 1))
    bins_ttft50.append(np.percentile([l.ttft_ms for l in succ], 50) if succ else 0)
    bins_ttft99.append(np.percentile([l.ttft_ms for l in succ], 99) if succ else 0)
    bins_tokens.append(sum(l.output_tokens for l in succ) / BIN_S)

x_hours = [b * BIN_S / 3600 for b in range(n_bins)]


fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.50, wspace=0.32)

deg_lo, deg_hi = 2.0, 2.5   # degradation window in hours

# Panel 1: request rate and error rate
ax  = fig.add_subplot(gs[0, 0])
ax2 = ax.twinx()
ax.plot(x_hours, bins_rps, color='#1f77b4', lw=1.5, label='Req/s')
ax2.plot(x_hours, [e*100 for e in bins_err], color='#d62728', lw=1.5, label='Error rate %')
ax.axvspan(deg_lo, deg_hi, alpha=0.12, color='red', label='Degradation')
ax.set_xlabel('Time (hours)', fontsize=10)
ax.set_ylabel('Requests / second', color='#1f77b4', fontsize=10)
ax2.set_ylabel('Error rate %', color='#d62728', fontsize=10)
ax.set_title('Request Rate and Error Rate', fontsize=11)
ax.grid(alpha=0.3)

# Panel 2: TTFT over time
ax = fig.add_subplot(gs[0, 1])
ax.plot(x_hours, bins_ttft50, color='#2ca02c', lw=1.5, label='P50')
ax.plot(x_hours, bins_ttft99, color='#ff7f0e', lw=1.5, label='P99')
ax.axhline(500, color='red', ls='--', lw=1.5, label='SLO threshold (500 ms)')
ax.axvspan(deg_lo, deg_hi, alpha=0.12, color='red')
ax.set_xlabel('Time (hours)', fontsize=10)
ax.set_ylabel('TTFT (ms)', fontsize=10)
ax.set_title('Time-to-First-Token Over Time', fontsize=11)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Panel 3: output token distribution
ax = fig.add_subplot(gs[1, 0])
out_toks = [l.output_tokens for l in hourly_logs if l.status == 'success']
ax.hist(out_toks, bins=50, color='#9467bd', alpha=0.75, edgecolor='white')
ax.axvline(np.median(out_toks), color='black', ls='--', lw=1.5,
           label=f'Median={int(np.median(out_toks))}')
ax.axvline(np.percentile(out_toks, 99), color='red', ls=':', lw=1.5,
           label=f'P99={int(np.percentile(out_toks, 99))}')
ax.set_xlabel('Output tokens per request', fontsize=10)
ax.set_ylabel('Count', fontsize=10)
ax.set_title('Output Token Distribution', fontsize=11)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Panel 4: SLO burn rate gauges
ax = fig.add_subplot(gs[1, 1])
# Recompute burn rates on the full hourly dataset
tracker_full = SLOTracker(slos[0], window_s=N_HOURS * 3600)
for log in hourly_logs:
    tracker_full.record(log)
br = tracker_full.burn_rate()

burn_names   = ['TTFT P99', 'TPOT P99', 'E2E P95']
burn_vals    = []
for slo, tracker in zip(slos, trackers):
    t2 = SLOTracker(slo, window_s=N_HOURS * 3600)
    for log in hourly_logs:
        t2.record(log)
    burn_vals.append(t2.burn_rate()['burn_rate'])

colors_b = ['#2ca02c' if v <= 1 else '#ff7f0e' if v <= 6 else '#d62728'
            for v in burn_vals]
bars = ax.barh(burn_names, burn_vals, color=colors_b, alpha=0.85)
ax.axvline(1,  color='gray', ls='--', lw=1.5, label='Burn rate = 1 (on track)')
ax.axvline(6,  color='orange', ls=':', lw=1.5, label='Burn rate = 6 (warning)')
ax.axvline(14, color='red',    ls=':', lw=1.5, label='Burn rate = 14 (critical)')
for bar, v in zip(bars, burn_vals):
    ax.text(v + 0.05, bar.get_y() + bar.get_height()/2,
            f'{v:.1f}', va='center', fontsize=10, fontweight='bold')
ax.set_xlabel('Burn rate', fontsize=10)
ax.set_title('SLO Burn Rates (higher = worse)', fontsize=11)
ax.legend(fontsize=7)
ax.grid(alpha=0.3, axis='x')

# Panel 5: error type breakdown
ax = fig.add_subplot(gs[2, 0])
err_types = defaultdict(int)
for l in hourly_logs:
    if l.error_type:
        err_types[l.error_type] += 1
err_types['timeout'] += sum(1 for l in hourly_logs if l.status == 'timeout')
if err_types:
    ax.pie(err_types.values(), labels=err_types.keys(), autopct='%1.0f%%',
           colors=['#d62728','#ff7f0e','#2ca02c','#1f77b4'])
ax.set_title('Error Type Breakdown', fontsize=11)

# Panel 6: throughput in tokens/s
ax = fig.add_subplot(gs[2, 1])
ax.fill_between(x_hours, bins_tokens, alpha=0.4, color='#1f77b4')
ax.plot(x_hours, bins_tokens, color='#1f77b4', lw=1.5)
ax.axvspan(deg_lo, deg_hi, alpha=0.12, color='red', label='Degradation')
ax.set_xlabel('Time (hours)', fontsize=10)
ax.set_ylabel('Output tokens / second', fontsize=10)
ax.set_title('Output Throughput Over Time', fontsize=11)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.suptitle('LLM Production Observability Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/11_observability.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Injected degradation at t=2h-2.5h is visible in TTFT P99 (Panel 2)')
print(f'and the error rate spike (Panel 1).')


## 4. Health Checks and Readiness Probes

A health-check endpoint allows load balancers and Kubernetes to detect and remove
unhealthy pods from the serving pool before they impact users.

```python
from fastapi import FastAPI, Response

app = FastAPI()

@app.get('/health')
async def health():
    """
    Liveness probe: returns 200 if the server process is alive.
    Load balancer calls this every ~5 seconds.
    """
    return {'status': 'ok'}

@app.get('/ready')
async def ready(response: Response):
    """
    Readiness probe: returns 200 only if the server can serve requests.
    Returns 503 if:
      - Model weights not yet loaded
      - Request queue is full (shed load)
      - Recent error rate > 50% (circuit breaker)
    """
    rates = collector.windowed_rate(window_s=60)
    if rates['error_rate'] > 0.50:
        response.status_code = 503
        return {'status': 'overloaded', 'error_rate': rates['error_rate']}
    return {'status': 'ready', 'rps': round(rates['rps'], 1)}

@app.get('/metrics')
async def metrics():
    """
    Prometheus text format metrics for scraping.
    Grafana reads this endpoint every 15 seconds.
    """
    s = collector.summary()
    lines = [
        f'llm_ttft_p50_ms {s["ttft_p50_ms"]:.1f}',
        f'llm_ttft_p99_ms {s["ttft_p99_ms"]:.1f}',
        f'llm_error_rate  {s["error_rate"]:.4f}',
        f'llm_total_requests {s["total_requests"]}',
        f'llm_output_tokens_total {s["total_output_toks"]}',
    ]
    return Response(content='\n'.join(lines), media_type='text/plain')
```

## 5. Exercises

1. **Prometheus text format**: Extend `MetricsCollector.summary()` to emit
   histogram bucket counts in Prometheus text format (the `_bucket` metric type).
   Test that the output is parseable by `prometheus_client.exposition.choose_encoder`.

2. **Distributed trace propagation**: Add a `TraceContext` dataclass with
   `trace_id` and `span_id`. Implement a `propagate_context(request)` function
   that extracts the W3C `traceparent` header and creates a child span.

3. **Circuit breaker**: Implement a circuit breaker that opens (rejects all requests
   with 503) when the error rate exceeds 30% for more than 30 consecutive seconds,
   then enters half-open state (allows one probe request per 10 seconds).
   Test it by injecting a burst of error responses.

4. **Cost allocation**: Extend `RequestLog` with a `cost_usd` field computed as
   `prompt_tokens * 0.0003 / 1000 + output_tokens * 0.0006 / 1000` (hypothetical
   pricing). Aggregate cost by user_id over the 4-hour simulation.
   Which user accounts for the highest cost?

5. **Anomaly detection**: Implement a simple z-score anomaly detector on TTFT.
   Maintain a 5-minute rolling mean and standard deviation. Flag any request
   whose TTFT is more than 3 standard deviations above the rolling mean.
   Measure the false positive rate and detection latency on the degradation event.

---

## 6. References

- [Google SRE Workbook (2018) -- Chapter 5: Alerting on SLOs](https://sre.google/workbook/alerting-on-slos/)
- [Prometheus Data Model and Exposition Format](https://prometheus.io/docs/concepts/data_model/)
- [OpenTelemetry Specification](https://opentelemetry.io/docs/specs/otel/)
- [vLLM Metrics Documentation](https://docs.vllm.ai/en/latest/serving/metrics.html)
